In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

In [2]:
CURRENT_SCRIPT_PATH = os.path.abspath(__file__)
ROOT_DIR = os.path.dirname(os.path.dirname(CURRENT_SCRIPT_PATH))
 
if os.path.basename(os.path.dirname(CURRENT_SCRIPT_PATH)) != "src":
    ROOT_DIR = os.path.dirname(CURRENT_SCRIPT_PATH)
 
DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
CHARTS_DIR = os.path.join(DATA_DIR, "charts")
 
for folder in [RAW_DIR, PROCESSED_DIR, CHARTS_DIR]:
    os.makedirs(folder, exist_ok=True)
 
PR_INPUT_FILE = os.path.join(RAW_DIR, "agile_pr_level.csv")
PR_OUTPUT_FILE = os.path.join(PROCESSED_DIR, "analyzed_pr_data.csv")
CHARTS_OUTPUT_DIR = CHARTS_DIR

NameError: name '__file__' is not defined

In [ ]:
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

In [ ]:
def load_data(filepath):
    if not os.path.exists(filepath):
        print(f"Fichier non trouve: {filepath}")
        return None
    return pd.read_csv(filepath)

In [ ]:
def analyze_story_points(df):
    print("\n" + "="*80)
    print("ANALYSE - STORY POINTS (CIBLE)")
    print("="*80)
    
    print(f"\nComte: {len(df['story_points'])}")
    print(f"Moyenne: {df['story_points'].mean():.2f}")
    print(f"Mediane: {df['story_points'].median():.2f}")
    print(f"Ecart-type: {df['story_points'].std():.2f}")
    print(f"Min: {df['story_points'].min():.0f}")
    print(f"Max: {df['story_points'].max():.0f}")
    print(f"Q1: {df['story_points'].quantile(0.25):.2f}")
    print(f"Q3: {df['story_points'].quantile(0.75):.2f}")
    print(f"IQR: {df['story_points'].quantile(0.75) - df['story_points'].quantile(0.25):.2f}")
    
    print("\nDistribution des valeurs:")
    print(df['story_points'].value_counts().sort_index())
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    axes[0, 0].hist(df['story_points'], bins=10, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(df['story_points'].mean(), color='red', linestyle='--', linewidth=2, label=f"Moyenne: {df['story_points'].mean():.2f}")
    axes[0, 0].axvline(df['story_points'].median(), color='green', linestyle='--', linewidth=2, label=f"Mediane: {df['story_points'].median():.2f}")
    axes[0, 0].set_xlabel('Story Points')
    axes[0, 0].set_ylabel('Frequence')
    axes[0, 0].set_title('Distribution des Story Points (Histogramme)')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    bp = axes[0, 1].boxplot(df['story_points'], vert=True, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
    axes[0, 1].set_ylabel('Story Points')
    axes[0, 1].set_title('Distribution des Story Points (Box Plot)')
    axes[0, 1].grid(alpha=0.3, axis='y')
    
    sp_counts = df['story_points'].value_counts().sort_index()
    axes[1, 0].bar(sp_counts.index, sp_counts.values, color='coral', edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('Story Points')
    axes[1, 0].set_ylabel('Nombre de PRs')
    axes[1, 0].set_title('Frequence des Valeurs de Story Points')
    axes[1, 0].set_xticks(sorted(df['story_points'].unique()))
    axes[1, 0].grid(alpha=0.3, axis='y')
    
    sorted_sp = sorted(df['story_points'])
    cumulative = np.arange(1, len(sorted_sp) + 1) / len(sorted_sp) * 100
    axes[1, 1].plot(sorted_sp, cumulative, marker='o', linestyle='-', linewidth=2, markersize=8, color='darkblue')
    axes[1, 1].set_xlabel('Story Points')
    axes[1, 1].set_ylabel('Percentile (%)')
    axes[1, 1].set_title('Distribution Cumulative des Story Points')
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS_OUTPUT_DIR, "01_story_points_distribution.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\nGraphique sauvegarde: 01_story_points_distribution.png")
 

In [ ]:
def analyze_by_language_and_repo(df):
    print("\n" + "="*80)
    print("ANALYSE - STORY POINTS PAR LANGUE ET REPOSITORY")
    print("="*80)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    langue_stats = df.groupby('language')['story_points'].agg(['count', 'mean', 'min', 'max', 'std'])
    print("\nStory Points par Langue:")
    print(langue_stats.to_string())
    
    colors_langs = plt.cm.Set3(np.linspace(0, 1, len(langue_stats)))
    axes[0].bar(langue_stats.index, langue_stats['mean'], color=colors_langs, edgecolor='black', alpha=0.8)
    axes[0].errorbar(langue_stats.index, langue_stats['mean'], yerr=langue_stats['std'], 
                     fmt='none', color='black', capsize=5, capthick=2, linewidth=2)
    axes[0].set_ylabel('Story Points (moyenne)')
    axes[0].set_title('Story Points Moyens par Langue')
    axes[0].grid(alpha=0.3, axis='y')
    
    repo_stats = df.groupby('repo_name')['story_points'].agg(['count', 'mean', 'min', 'max', 'std']).sort_values('mean')
    print("\nStory Points par Repository:")
    print(repo_stats.to_string())
    
    colors_repos = plt.cm.Spectral(np.linspace(0, 1, len(repo_stats)))
    axes[1].barh(repo_stats.index, repo_stats['mean'], color=colors_repos, edgecolor='black', alpha=0.8)
    axes[1].errorbar(repo_stats['mean'], repo_stats.index, xerr=repo_stats['std'], 
                     fmt='none', color='black', capsize=5, capthick=2, linewidth=2)
    axes[1].set_xlabel('Story Points (moyenne)')
    axes[1].set_title('Story Points Moyens par Repository')
    axes[1].grid(alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS_OUTPUT_DIR, "02_story_points_by_language_repo.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\nGraphique sauvegarde: 02_story_points_by_language_repo.png")

In [ ]:
def analyze_ai_assistance(df):
    print("\n" + "="*80)
    print("ANALYSE - ASSISTANCE IA")
    print("="*80)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    ai_counts = df['is_ai_assisted'].value_counts()
    print(f"\nTotal PRs: {len(df)}")
    print(f"PRs assistees par IA: {df['is_ai_assisted'].sum()} ({100*df['is_ai_assisted'].sum()/len(df):.1f}%)")
    print(f"PRs non assistees: {(~df['is_ai_assisted'].astype(bool)).sum()} ({100*(~df['is_ai_assisted'].astype(bool)).sum()/len(df):.1f}%)")
    
    colors_ai = ['#ff9999', '#66b3ff']
    explode = (0.05, 0)
    wedges, texts, autotexts = axes[0, 0].pie(ai_counts.values, 
                                                 labels=['Non assiste', 'Assiste par IA'],
                                                 autopct='%1.1f%%',
                                                 colors=colors_ai,
                                                 explode=explode,
                                                 startangle=90,
                                                 textprops={'fontsize': 11})
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    axes[0, 0].set_title('Proportion de PRs Assistees par IA')
    
    ai_sp_stats = df.groupby('is_ai_assisted')['story_points'].agg(['count', 'mean', 'median', 'std'])
    print(f"\nStory Points par Assistance IA:")
    print(ai_sp_stats.to_string())
    
    bp = axes[0, 1].boxplot([df[df['is_ai_assisted']==0]['story_points'],
                              df[df['is_ai_assisted']==1]['story_points']],
                            labels=['Non assiste', 'Assiste par IA'],
                            patch_artist=True)
    for patch, color in zip(bp['boxes'], colors_ai):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[0, 1].set_ylabel('Story Points')
    axes[0, 1].set_title('Story Points selon Assistance IA')
    axes[0, 1].grid(alpha=0.3, axis='y')
    
    print(f"\nAI Generation Ratio:")
    print(df['ai_generation_ratio'].describe().to_string())
    
    axes[1, 0].hist(df['ai_generation_ratio'].dropna(), bins=15, color='mediumseagreen', edgecolor='black', alpha=0.7)
    axes[1, 0].axvline(df['ai_generation_ratio'].mean(), color='red', linestyle='--', linewidth=2, 
                       label=f"Moyenne: {df['ai_generation_ratio'].mean():.2f}")
    axes[1, 0].set_xlabel('AI Generation Ratio')
    axes[1, 0].set_ylabel('Frequence')
    axes[1, 0].set_title('Distribution du Ratio de Generation IA')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    
    scatter = axes[1, 1].scatter(df['ai_mentions'], df['ai_generation_ratio'], 
                                  c=df['story_points'], cmap='viridis', s=150, alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Mentions d IA')
    axes[1, 1].set_ylabel('Ratio de Generation IA')
    axes[1, 1].set_title('Mentions IA vs Ratio de Generation (couleur = story points)')
    cbar = plt.colorbar(scatter, ax=axes[1, 1])
    cbar.set_label('Story Points')
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS_OUTPUT_DIR, "03_ai_assistance_analysis.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\nGraphique sauvegarde: 03_ai_assistance_analysis.png")
 
def analyze_code_quality(df):
    print("\n" + "="*80)
    print("ANALYSE - QUALITE DU CODE")
    print("="*80)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    print(f"\nPost Coding Churn:")
    print(df['post_coding_churn'].describe().to_string())
    
    axes[0, 0].hist(df['post_coding_churn'].dropna(), bins=15, color='salmon', edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(df['post_coding_churn'].mean(), color='red', linestyle='--', linewidth=2)
    axes[0, 0].set_xlabel('Post Coding Churn')
    axes[0, 0].set_ylabel('Frequence')
    axes[0, 0].set_title('Distribution - Post Coding Churn')
    axes[0, 0].grid(alpha=0.3)
    
    print(f"\nCyclomatic Complexity:")
    print(df['post_coding_cyclomatic_proxy'].describe().to_string())
    
    axes[0, 1].hist(df['post_coding_cyclomatic_proxy'].dropna(), bins=15, color='lightblue', edgecolor='black', alpha=0.7)
    axes[0, 1].axvline(df['post_coding_cyclomatic_proxy'].mean(), color='red', linestyle='--', linewidth=2)
    axes[0, 1].set_xlabel('Complexite Cyclomatique')
    axes[0, 1].set_ylabel('Frequence')
    axes[0, 1].set_title('Distribution - Complexite Cyclomatique')
    axes[0, 1].grid(alpha=0.3)
    
    print(f"\nTest Coverage Ratio:")
    print(df['post_coding_test_coverage_ratio'].describe().to_string())
    
    axes[0, 2].hist(df['post_coding_test_coverage_ratio'].dropna(), bins=15, color='lightgreen', edgecolor='black', alpha=0.7)
    axes[0, 2].axvline(df['post_coding_test_coverage_ratio'].mean(), color='red', linestyle='--', linewidth=2)
    axes[0, 2].set_xlabel('Ratio de Couverture')
    axes[0, 2].set_ylabel('Frequence')
    axes[0, 2].set_title('Distribution - Couverture de Tests')
    axes[0, 2].grid(alpha=0.3)
    
    axes[1, 0].scatter(df['story_points'], df['post_coding_churn'], s=150, alpha=0.6, color='salmon', edgecolor='black')
    z = np.polyfit(df['story_points'], df['post_coding_churn'], 1)
    p = np.poly1d(z)
    axes[1, 0].plot(sorted(df['story_points']), p(sorted(df['story_points'])), "r--", linewidth=2, label='Trend')
    axes[1, 0].set_xlabel('Story Points')
    axes[1, 0].set_ylabel('Post Coding Churn')
    axes[1, 0].set_title('Story Points vs Churn')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    
    axes[1, 1].scatter(df['story_points'], df['post_coding_cyclomatic_proxy'], s=150, alpha=0.6, color='lightblue', edgecolor='black')
    z = np.polyfit(df['story_points'], df['post_coding_cyclomatic_proxy'], 1)
    p = np.poly1d(z)
    axes[1, 1].plot(sorted(df['story_points']), p(sorted(df['story_points'])), "r--", linewidth=2, label='Trend')
    axes[1, 1].set_xlabel('Story Points')
    axes[1, 1].set_ylabel('Complexite Cyclomatique')
    axes[1, 1].set_title('Story Points vs Complexite')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)
    
    axes[1, 2].scatter(df['story_points'], df['post_coding_test_coverage_ratio'], s=150, alpha=0.6, color='lightgreen', edgecolor='black')
    z = np.polyfit(df['story_points'], df['post_coding_test_coverage_ratio'], 1)
    p = np.poly1d(z)
    axes[1, 2].plot(sorted(df['story_points']), p(sorted(df['story_points'])), "r--", linewidth=2, label='Trend')
    axes[1, 2].set_xlabel('Story Points')
    axes[1, 2].set_ylabel('Couverture de Tests')
    axes[1, 2].set_title('Story Points vs Couverture Tests')
    axes[1, 2].legend()
    axes[1, 2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS_OUTPUT_DIR, "04_code_quality_analysis.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\nGraphique sauvegarde: 04_code_quality_analysis.png")

In [ ]:
def analyze_correlation(df):
    print("\n" + "="*80)
    print("ANALYSE - MATRICE DE CORRELATION")
    print("="*80)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    correlation_matrix = df[numeric_cols].corr()
    
    print(f"\nCorrelations avec STORY_POINTS:")
    correlations_sp = correlation_matrix['story_points'].sort_values(ascending=False)
    print(correlations_sp.to_string())
    
    plt.figure(figsize=(16, 12))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8}, 
                annot_kws={'fontsize': 10})
    plt.title('Matrice de Correlation - Toutes Variables Numeriques')
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS_OUTPUT_DIR, "05_correlation_matrix.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\nGraphique sauvegarde: 05_correlation_matrix.png")

In [ ]:
def analyze_collaboration(df):
    print("\n" + "="*80)
    print("ANALYSE - COLLABORATION ET EXPERIENCE")
    print("="*80)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    print(f"\nParticipants a la Discussion:")
    print(df['pre_coding_discussion_participants'].describe().to_string())
    
    axes[0, 0].hist(df['pre_coding_discussion_participants'].dropna(), bins=10, color='skyblue', edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(df['pre_coding_discussion_participants'].mean(), color='red', linestyle='--', linewidth=2)
    axes[0, 0].set_xlabel('Nombre de Participants')
    axes[0, 0].set_ylabel('Frequence')
    axes[0, 0].set_title('Distribution - Participants a la Discussion')
    axes[0, 0].grid(alpha=0.3)
    
    print(f"\nAnciennete de l'Auteur (jours):")
    print(df['pre_coding_author_tenure_days'].describe().to_string())
    
    axes[0, 1].hist(df['pre_coding_author_tenure_days'].dropna(), bins=10, color='lightcoral', edgecolor='black', alpha=0.7)
    axes[0, 1].axvline(df['pre_coding_author_tenure_days'].mean(), color='red', linestyle='--', linewidth=2)
    axes[0, 1].set_xlabel('Jours')
    axes[0, 1].set_ylabel('Frequence')
    axes[0, 1].set_title('Distribution - Anciennete de l Auteur')
    axes[0, 1].grid(alpha=0.3)
    
    axes[1, 0].scatter(df['pre_coding_discussion_participants'], df['story_points'], s=150, alpha=0.6, color='skyblue', edgecolor='black')
    z = np.polyfit(df['pre_coding_discussion_participants'], df['story_points'], 1)
    p = np.poly1d(z)
    axes[1, 0].plot(sorted(df['pre_coding_discussion_participants']), p(sorted(df['pre_coding_discussion_participants'])), "r--", linewidth=2)
    axes[1, 0].set_xlabel('Participants a la Discussion')
    axes[1, 0].set_ylabel('Story Points')
    axes[1, 0].set_title('Story Points vs Participants')
    axes[1, 0].grid(alpha=0.3)
    
    axes[1, 1].scatter(df['pre_coding_author_tenure_days'], df['story_points'], s=150, alpha=0.6, color='lightcoral', edgecolor='black')
    z = np.polyfit(df['pre_coding_author_tenure_days'], df['story_points'], 1)
    p = np.poly1d(z)
    axes[1, 1].plot(sorted(df['pre_coding_author_tenure_days']), p(sorted(df['pre_coding_author_tenure_days'])), "r--", linewidth=2)
    axes[1, 1].set_xlabel('Anciennete de l Auteur (jours)')
    axes[1, 1].set_ylabel('Story Points')
    axes[1, 1].set_title('Story Points vs Anciennete')
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS_OUTPUT_DIR, "06_collaboration_analysis.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\nGraphique sauvegarde: 06_collaboration_analysis.png")
 
def print_summary(df):
    print("\n" + "="*80)
    print("RESUME EXECUTIF DE L'ANALYSE")
    print("="*80)
    
    print(f"\nDATASET:")
    print(f"   Total PRs: {len(df)}")
    print(f"   Langages: {df['language'].nunique()} - {', '.join(df['language'].unique())}")
    print(f"   Repositories: {df['repo_name'].nunique()}")
    
    print(f"\nSTORY POINTS (CIBLE):")
    print(f"   Moyenne: {df['story_points'].mean():.2f}")
    print(f"   Mediane: {df['story_points'].median():.2f}")
    print(f"   Ecart-type: {df['story_points'].std():.2f}")
    print(f"   Range: [{df['story_points'].min():.0f}, {df['story_points'].max():.0f}]")
    
    print(f"\nASSISTANCE IA:")
    ai_pct = 100*df['is_ai_assisted'].sum()/len(df)
    print(f"   PRs assistees: {df['is_ai_assisted'].sum()} / {len(df)} ({ai_pct:.1f}%)")
    print(f"   Mentions IA moyenne: {df['ai_mentions'].mean():.2f}")
    print(f"   Generation Ratio moyenne: {df['ai_generation_ratio'].mean():.2f}")
    print(f"   Speed Proxy moyenne: {df['ai_speed_proxy'].mean():.2f}")
    
    print(f"\nQUALITE DU CODE:")
    print(f"   Post Coding Churn (moyenne): {df['post_coding_churn'].mean():.2f}")
    print(f"   Complexite Cyclomatique (moyenne): {df['post_coding_cyclomatic_proxy'].mean():.2f}")
    print(f"   Couverture de Tests (moyenne): {df['post_coding_test_coverage_ratio'].mean():.4f}")
    
    print(f"\nCOLLABORATION:")
    print(f"   Participants (moyenne): {df['pre_coding_discussion_participants'].mean():.2f}")
    print(f"   Anciennete auteur (jours): {df['pre_coding_author_tenure_days'].mean():.0f}")
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    correlation_matrix = df[numeric_cols].corr()
    print(f"\nTOP 5 CORRELATIONS AVEC STORY POINTS:")
    top_corr = correlation_matrix['story_points'].abs().sort_values(ascending=False)[1:6]
    for col, val in top_corr.items():
        print(f"   {col}: {val:.3f}")
    
    print(f"\n" + "="*80)
    print(f"ANALYSE TERMINEEE!")
    print(f"="*80)

In [ ]:
def main():
    print("--- DEMARRAGE DE L'ANALYSE ---")
    
    df = load_data(PR_INPUT_FILE)
    if df is None:
        print("Charger votre fichier CSV dans le repertoire raw/")
        return
    
    print(f"\nFichier charge avec succes: {len(df)} lignes")
    
    analyze_story_points(df)
    analyze_by_language_and_repo(df)
    analyze_ai_assistance(df)
    analyze_code_quality(df)
    analyze_correlation(df)
    analyze_collaboration(df)
    print_summary(df)
    
    df.to_csv(PR_OUTPUT_FILE, index=False, encoding='utf-8')
    print(f"\nFichier analyse sauvegarde: {PR_OUTPUT_FILE}")
    print(f"Graphiques sauvegardes dans: {CHARTS_OUTPUT_DIR}")
 
if __name__ == "__main__":
    main()
